# Guided walkthrough: the "Add to cart" button test

Read the main `README.md` first. It has the scenario, the facts you can rely on, and the data dictionary.

Run the cells in order. The notebook asks you to make decisions along the way. Write them in `plan_template.md` or in the markdown cells.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

DATA = Path("../data")

## Step 1: Plan before you look at any data

Decide, in writing, how you will judge this test.

The decisions you need to make:

1. What is your **metric**, and what is one row of data for it?
2. Which **test** will you use?
3. What is the **smallest difference** between the buttons that would justify rolling out the redesign?
4. What **significance level** and **power** will you use?
5. How many users do you need, and **how many days** will that take?
6. What will you do for each possible result?

The helper below answers question 5 for a click-rate comparison. It tells you how many users each group needs to detect a given difference with the chosen significance level and power.

In [ ]:
def users_needed_per_arm(baseline, lift, alpha=0.05, power=0.80):
    """Users needed in EACH group to detect an absolute change of `lift`
    with a two-sided two-proportion z-test."""
    p1, p2 = baseline, baseline + lift
    p_bar = (p1 + p2) / 2
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_power = stats.norm.ppf(power)
    top = (z_alpha * np.sqrt(2 * p_bar * (1 - p_bar)) + z_power * np.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2
    return top / (p2 - p1) ** 2


baseline = 0.10           # from the README: about 10% of users clicked the current button
new_users_per_day = 800   # from the README

rows = []
for lift in [0.005, 0.01, 0.015, 0.02, 0.03]:
    n = users_needed_per_arm(baseline, lift)
    rows.append({
        "difference to detect": f"{lift * 100:.1f} points",
        "users per group": round(n),
        "days needed": round(2 * n / new_users_per_day, 1),
    })
pd.DataFrame(rows)

Which of these differences can you detect within the two-week limit? What does that mean for the plan you write?

**Fill in `plan_template.md` now, before Step 2.**

## Step 2: Load the first five days

In [ ]:
early = pd.read_csv(DATA / "experiment_days_1-5.csv", parse_dates=["timestamp"])
early.head()

In [ ]:
print("Rows (visits):", len(early))
print("Distinct users:", early["user_id"].nunique())
early.groupby("button")["user_id"].nunique()

Before comparing click rates, check that the groups are about the size you expect. Users were assigned 50/50, so the two group sizes should be close. A large gap would suggest something is wrong with the assignment. A chi-square goodness-of-fit test on the group sizes is a quick check.

In [ ]:
group_sizes = early.groupby("button")["user_id"].nunique()
stats.chisquare(group_sizes)

## Step 3: One row per user

The data has one row per **visit**, but the people who were randomly assigned a button are **users**. A user who visits four times is one person, not four independent observations.

Collapse the visits so that each user is a single row: which button they saw, and whether they clicked at least once.

In [ ]:
def per_user(visits):
    return visits.groupby("user_id").agg(
        button=("button", "first"),
        clicked=("clicked", "max"),
        visits=("visit_id", "count"),
    )


users = per_user(early)
users.head()

In [ ]:
summary = users.groupby("button").agg(users=("clicked", "size"), clicked=("clicked", "sum"))
summary["click_rate"] = summary["clicked"] / summary["users"]
summary

## Step 4: Test the difference

Now it is the two-group comparison from your statistics class: two groups of users, each either clicked or did not.

- Null hypothesis: the click rate is the same for A and B.
- Alternative hypothesis: the click rates differ.

In [ ]:
def two_proportion_test(x_a, n_a, x_b, n_b, alpha=0.05):
    p_a, p_b = x_a / n_a, x_b / n_b
    p_pool = (x_a + x_b) / (n_a + n_b)
    se_pool = np.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
    z = (p_a - p_b) / se_pool
    p_value = 2 * stats.norm.sf(abs(z))
    se_diff = np.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
    margin = stats.norm.ppf(1 - alpha / 2) * se_diff
    return {
        "rate_A": p_a, "rate_B": p_b, "difference (A - B)": p_a - p_b,
        "z": z, "p_value": p_value,
        "CI low": p_a - p_b - margin, "CI high": p_a - p_b + margin,
    }


def test_users(user_table):
    a = user_table[user_table["button"] == "A"]
    b = user_table[user_table["button"] == "B"]
    return two_proportion_test(a["clicked"].sum(), len(a), b["clicked"].sum(), len(b))


pd.Series(test_users(users)).round(4)

## Step 5: What do you tell the product manager today?

The product manager has only these five days of data and wants a decision.

Write down:

- What the result is, in plain language.
- What you recommend: roll out A, keep B, or keep running the test.
- How your recommendation follows from the plan you wrote in Step 1.

Do this before moving on.

## Step 6: All 14 days

Load the rest of the data. Your facilitator will tell you which days 6 to 14 file to start with. Later you can change `SET` and run the notebook from this step again.

Users who first appeared in days 1 to 5 and came back later show up in both files. Grouping by `user_id` handles that for you.

In [ ]:
SET = "set1"   # use the file your facilitator tells you to; try "set2" later

later = pd.read_csv(DATA / f"experiment_days_6-14_{SET}.csv", parse_dates=["timestamp"])
full = pd.concat([early, later], ignore_index=True)
full_users = per_user(full)

print("Distinct users:", len(full_users))
pd.Series(test_users(full_users)).round(4)

## Step 7: Look back, day by day

The table below shows what the analysis would have said at the end of each day, using only the data available up to that day.

In [ ]:
start = pd.Timestamp("2026-03-02")
full["day"] = (full["timestamp"] - start).dt.days + 1

rows = []
for d in range(1, 15):
    seen = per_user(full[full["day"] <= d])
    res = test_users(seen)
    rows.append({
        "through day": d, "users": len(seen),
        "rate A": res["rate_A"], "rate B": res["rate_B"], "p-value": res["p_value"],
    })
pd.DataFrame(rows).round(4)

## Reflect

- How did your day-5 recommendation compare with what you see after 14 days?
- How did your significance level change what you would have done on day 5?
- Switch `SET` in Step 6 to the other file and run from there. What changes, and what does that tell you about what day 5 could and could not show?